# Clasificadores CNN por transferencia

Fold 1 de José con EfficientNet-B0, ResNet50 y VGG16 y el protocolo elegido: ajuste completo, pesos por clase, tasa de aprendizaje 1e-4 y duración mínima de 5 s. Se entrena sin los recortes sin caja y las métricas se calculan solo sobre los eventos de José.

Antes de correr: Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4). Correr las tres celdas en orden.

In [ ]:
# datos y script: el zip ya esta en Drive y el script se baja fijo al commit a85cc3b
import os
from google.colab import drive
drive.mount('/content/drive')
SALIDA = '/content/drive/MyDrive/tesis/resultados_cnn'
DATOS = '/content/clasificacion_colab'
URL = 'https://raw.githubusercontent.com/Pattoxd45/tesis-sismovolcanica/a85cc3b9790326370aee74245c2db386d95e2644/scripts/clasificacion'
# se copia al disco de Colab: leer miles de imagenes directo desde Drive es lento
if not os.path.exists(DATOS):
    !cp /content/drive/MyDrive/tesis/clasificacion_colab.zip /content/paquete.zip && unzip -q -o /content/paquete.zip -d /content/ && rm /content/paquete.zip
!wget -q -O /content/entrenar_cnn.py {URL}/entrenar_cnn.py && wget -q -O /content/resumen_cnn.py {URL}/resumen_cnn.py
SCRIPT = '/content/entrenar_cnn.py'
!grep -c sin-falsas-alarmas {SCRIPT}
!ls {DATOS}/imagenes | wc -l
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# fold 1 con las tres redes; la que ya tiene su JSON en Drive se salta (si se corta, se vuelve a correr la celda)
PROTOCOLO = '--ajuste-completo --pesos-clase --lr 0.0001 --duracion-minima 5 --sin-falsas-alarmas'
for modelo in ['efficientnet_b0', 'resnet50', 'vgg16']:
    resultado = f'{SALIDA}/{modelo}_fold1_estirar_pesos_completo_lr0.0001_min5s_sin_falsas.json'
    if os.path.exists(resultado):
        print(modelo, 'ya hecho')
        continue
    !python {SCRIPT} --datos {DATOS} --modelo {modelo} --fold 1 --epocas 10 --salida {SALIDA} {PROTOCOLO}
    # si la GPU no alcanza, se repite con el lote en dos partes: mismo lote efectivo y misma perdida
    if not os.path.exists(resultado):
        !python {SCRIPT} --datos {DATOS} --modelo {modelo} --fold 1 --epocas 10 --salida {SALIDA} {PROTOCOLO} --acumular 2

In [ ]:
# tabla de las tres redes (solo los resultados con el protocolo nuevo)
!python /content/resumen_cnn.py {SALIDA}